In [11]:
"""
Exponential Mass Generation Model with CKM Matrix

This module implements a mass generation approach using an exponential function
of geodesic length (L), including all six quarks and CKM matrix calculations.

Author: Manus AI / Modified by Assistant
Date: April 29, 2025
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

class ExponentialMassModel:
    """
    An implementation of an exponential mass generation approach (mass = A * exp(k*L))
    that includes all six quarks and CKM matrix calculations.
    """

    def __init__(self):
        """
        Initialize the exponential model with lattice QCD parameters.
        """
        # Lattice QCD parameters (from provided papers and PDG)
        self.alpha_s_mz = 0.11803 # Strong coupling at Z-boson mass

        # Quark masses at reference scales (GeV)
        self.mc_mc = 1.2735 # Charm quark mass at its own scale
        self.mb_mb = 4.188 # Bottom quark mass at its own scale
        # PDG values for other quarks (GeV)
        self.mu_2gev = 0.00216 # Up quark mass at 2 GeV
        self.md_2gev = 0.00467 # Down quark mass at 2 GeV
        self.ms_2gev = 0.093 # Strange quark mass at 2 GeV
        self.mt_mt = 172.76 # Top quark mass at its own scale

        # Reference scales (GeV)
        self.mu_ref_light = 2.0 # Reference scale for light quarks (u, d, s)
        self.mu_c = self.mc_mc # Reference scale for charm quark
        self.mu_b = self.mb_mb # Reference scale for bottom quark
        self.mu_t = self.mt_mt # Reference scale for top quark
        self.mz = 91.1876 # Z-boson mass

        # --- Exponential parameters (to be optimized) ---
        # mass = A * exp(k*L)
        self.A_up = 1.37905363e-01 # Initial guess (from good vector)
        self.k_up = 2.02151405e+00   # Initial guess (from good vector)
        self.A_down = 1.50332792e-01 # Initial guess (from good vector)
        self.k_down = 1.36356449e+00   # Initial guess (from good vector)

        # Initialize geodesic lengths (to be optimized)
        self.L_u = 1.00000000e-03 # Initial guess (from good vector)
        self.L_d = 5.00000000e-01 # Initial guess (from good vector)
        self.L_s = 1.00000000e-01 # Initial guess (from good vector)
        self.L_c = 1.40209557e+00 # Initial guess (from good vector)
        self.L_b = 2.25322232e+00 # Initial guess (from good vector)
        self.L_t = 3.40261219e+00 # Initial guess (from good vector)

        # Initialize geodesic angles for CKM matrix (to be optimized)
        self.theta_12 = 2.26936877e-01 # Initial guess (from good vector)
        self.theta_13 = 3.69004328e-03 # Initial guess (from good vector)
        self.theta_23 = 4.18323275e-02 # Initial guess (from good vector)
        self.delta_cp = 1.14398379e+00 # Initial guess (from good vector)

        # Experimental CKM matrix magnitudes (PDG 2022)
        self.ckm_exp = np.array([
            [0.97435, 0.22500, 0.00369],
            [0.22486, 0.97349, 0.04182],
            [0.00857, 0.04110, 0.99915]
        ])

        # Beta function coefficients (unchanged)
        self.beta0_nf3 = (11.0 - 2.0/3.0 * 3.0) / 4.0
        self.beta1_nf3 = (102.0 - 38.0/3.0 * 3.0) / 16.0
        self.beta0_nf4 = (11.0 - 2.0/3.0 * 4.0) / 4.0
        self.beta1_nf4 = (102.0 - 38.0/3.0 * 4.0) / 16.0
        self.beta0_nf5 = (11.0 - 2.0/3.0 * 5.0) / 4.0
        self.beta1_nf5 = (102.0 - 38.0/3.0 * 5.0) / 16.0
        self.beta0_nf6 = (11.0 - 2.0/3.0 * 6.0) / 4.0
        self.beta1_nf6 = (102.0 - 38.0/3.0 * 6.0) / 16.0

        # Anomalous dimension coefficients (unchanged)
        self.gamma0 = 1.0
        self.gamma1_nf3 = (202.0/3.0 - 20.0/9.0 * 3.0) / 16.0
        self.gamma1_nf4 = (202.0/3.0 - 20.0/9.0 * 4.0) / 16.0
        self.gamma1_nf5 = (202.0/3.0 - 20.0/9.0 * 5.0) / 16.0
        self.gamma1_nf6 = (202.0/3.0 - 20.0/9.0 * 6.0) / 16.0

        # Generation scaling factors (to be optimized)
        self.gen_scale = [1.56969719e-02, 5.42245494e-01, 1.28994060e+00] # Initial guess (from good vector)

        # Optimization status
        self.is_optimized = False
        self.results = {}

        # Quark information (unchanged)
        self.quark_info = {
            "u": {"type": "up", "generation": 1, "ref_mass": self.mu_2gev, "ref_scale": self.mu_ref_light},
            "d": {"type": "down", "generation": 1, "ref_mass": self.md_2gev, "ref_scale": self.mu_ref_light},
            "s": {"type": "down", "generation": 2, "ref_mass": self.ms_2gev, "ref_scale": self.mu_ref_light},
            "c": {"type": "up", "generation": 2, "ref_mass": self.mc_mc, "ref_scale": self.mu_c},
            "b": {"type": "down", "generation": 3, "ref_mass": self.mb_mb, "ref_scale": self.mu_b},
            "t": {"type": "up", "generation": 3, "ref_mass": self.mt_mt, "ref_scale": self.mu_t}
        }

    def alpha_s(self, mu):
        """Calculate the strong coupling constant at scale mu using 1-loop running (simplified)."""
        # Determine number of active flavors
        if mu < 1.3: nf = 3; beta0 = self.beta0_nf3
        elif mu < 4.2: nf = 4; beta0 = self.beta0_nf4
        elif mu < 173.0: nf = 5; beta0 = self.beta0_nf5
        else: nf = 6; beta0 = self.beta0_nf6

        if abs(mu - self.mz) < 0.1: return self.alpha_s_mz

        t = np.log(mu**2 / self.mz**2)

        # Determine beta0 based on the scale mu relative to mz for running
        if mu >= self.mz:
            beta0_run = beta0
        else: # mu < mz
            beta0_run = self.beta0_nf5 # Running down from mz (nf=5)

        if abs(beta0_run) < 1e-9: return self.alpha_s_mz

        alpha_s_inv = 1.0 / self.alpha_s_mz + beta0_run * t / (2 * np.pi)

        if alpha_s_inv <= 1e-6: return 10.0 # Cap at Landau pole

        alpha_s_val = 1.0 / alpha_s_inv

        # Freeze alpha_s at low scales (e.g., below 1 GeV) - optional
        if mu < 1.0:
             t_freeze = np.log(1.0**2 / self.mz**2)
             beta0_freeze = self.beta0_nf5 # Running down from mz to 1 GeV
             alpha_s_inv_freeze = 1.0 / self.alpha_s_mz + beta0_freeze * t_freeze / (2 * np.pi)
             if alpha_s_inv_freeze <= 1e-6: return 10.0 # Cap
             return 1.0 / alpha_s_inv_freeze

        return max(0.01, alpha_s_val)

    def running_mass(self, m_ref, mu_ref, mu, nf):
        """Calculate the running mass at scale mu using 1-loop RGE."""
        if nf == 3: gamma0 = self.gamma0; beta0 = self.beta0_nf3
        elif nf == 4: gamma0 = self.gamma0; beta0 = self.beta0_nf4
        elif nf == 5: gamma0 = self.gamma0; beta0 = self.beta0_nf5
        else: gamma0 = self.gamma0; beta0 = self.beta0_nf6

        if abs(mu - mu_ref) < 0.01: return m_ref

        mu_safe = max(mu, 0.1)
        mu_ref_safe = max(mu_ref, 0.1)

        try:
            alpha_ref = self.alpha_s(mu_ref_safe)
            alpha_mu = self.alpha_s(mu_safe)
            alpha_ref = max(alpha_ref, 1e-6)
            alpha_mu = max(alpha_mu, 1e-6)
            if not (np.isfinite(alpha_ref) and np.isfinite(alpha_mu)):
                raise ValueError("alpha_s non-finite")
        except (ValueError, OverflowError) as e:
            print(f"Warning: alpha_s calc failed ({mu_ref_safe} -> {mu_safe}). Ret ref mass. Err: {e}")
            return m_ref

        try:
            if abs(beta0) < 1e-9: power = 0
            else: power = gamma0 / (2 * beta0)
            ratio = alpha_mu / alpha_ref

            if ratio <= 0:
                 print(f"Warning: alpha_s ratio <= 0 ({ratio:.2e}). Ret ref mass.")
                 m_mu = m_ref
            else:
                 exponent = power * np.log(ratio)
                 if exponent > 700: # Avoid exp overflow
                     print(f"Warning: Exp overflow predicted. Ret ref mass.")
                     m_mu = m_ref
                 else:
                     m_mu = m_ref * (ratio**power)

            if not np.isfinite(m_mu) or m_mu < 0:
                print(f"Warning: Running mass non-finite/neg ({m_mu:.2e}). Ret ref mass.")
                return m_ref
            return max(m_mu, 1e-9)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            print(f"Warning: Running mass calc error: {e}. Ret ref mass.")
            return m_ref

    def calculate_ckm_matrix(self):
        """Calculate the CKM matrix using the standard parameterization."""
        s12, c12 = np.sin(self.theta_12), np.cos(self.theta_12)
        s13, c13 = np.sin(self.theta_13), np.cos(self.theta_13)
        s23, c23 = np.sin(self.theta_23), np.cos(self.theta_23)
        delta = self.delta_cp
        exp_neg_id = np.exp(-1j * delta)
        exp_id = np.exp(1j * delta)

        ckm = np.array([
            [c12 * c13, s12 * c13, s13 * exp_neg_id],
            [-s12 * c23 - c12 * s23 * s13 * exp_id, c12 * c23 - s12 * s23 * s13 * exp_id, s23 * c13],
            [s12 * s23 - c12 * c23 * s13 * exp_id, -c12 * s23 - s12 * c23 * s13 * exp_id, c23 * c13]
        ])
        return ckm

    def calculate_mass(self, quark, L=None):
        """
        Calculate quark mass from geodesic length using exponential approach.
        Uses mass = A * exp(k*L).
        """
        quark_type = self.quark_info[quark]["type"]
        generation = self.quark_info[quark]["generation"]

        if L is None:
            L = getattr(self, f"L_{quark}", 0.0)

        current_gen_scale = getattr(self, "gen_scale", [1.0, 1.0, 1.0])

        try:
            if quark_type == "up":
                A = getattr(self, "A_up", 0.0)
                k = getattr(self, "k_up", 0.0)
            else: # quark_type == "down"
                A = getattr(self, "A_down", 0.0)
                k = getattr(self, "k_down", 0.0)

            # Calculate base mass using exponential: A * exp(k*L)
            exponent = k * L
            if exponent > 700: # Prevent overflow in np.exp
                print(f"Warning: Exponent overflow predicted for {quark} (k*L = {exponent:.2f}). Returning large mass.")
                base_mass = 1e10 # Assign large mass instead of infinity
            else:
                base_mass = A * np.exp(exponent)

        except (OverflowError, ValueError) as e:
            print(f"Warning: Error calculating base mass for {quark}: {e}. Returning 1e-9.")
            base_mass = 1e-9

        # Apply generation-specific scaling
        if 0 <= generation - 1 < len(current_gen_scale):
             mass = base_mass * current_gen_scale[generation - 1]
        else:
             print(f"Warning: Invalid generation index {generation} for quark {quark}. Assigning large mass.")
             mass = 1e9

        # Ensure mass is positive and finite
        if not np.isfinite(mass):
            print(f"Warning: Calculated mass for quark {quark} is non-finite. Returning 1e-9.")
            return 1e-9

        return max(mass, 1e-9) # Ensure mass is positive

    def optimize_parameters(self):
        """
        Optimize model parameters (A, k, L, gen_scale, CKM) for exponential model.
        """
        def objective(params):
            # Unpack parameters safely
            try:
                # Number of parameters: 2(A_up, k_up) + 2(A_down, k_down) + 6(L) + 3(gen) + 4(CKM) = 17
                current_A_up = params[0]
                current_k_up = params[1]
                current_A_down = params[2]
                current_k_down = params[3]
                current_L_u = params[4]
                current_L_d = params[5]
                current_L_s = params[6]
                current_L_c = params[7]
                current_L_b = params[8]
                current_L_t = params[9]
                current_gen_scale = params[10:13] # Indices 10, 11, 12
                current_theta_12 = params[13]
                current_theta_13 = params[14]
                current_theta_23 = params[15]
                current_delta_cp = params[16]
            except IndexError:
                 print("Error: Incorrect number of parameters passed to objective function.")
                 return 1e20 # Return large error

            # Store original values to restore later
            original_params = {
                 "A_up": self.A_up, "k_up": self.k_up, "A_down": self.A_down, "k_down": self.k_down,
                 "L_u": self.L_u, "L_d": self.L_d, "L_s": self.L_s, "L_c": self.L_c, "L_b": self.L_b, "L_t": self.L_t,
                 "gen_scale": self.gen_scale, "theta_12": self.theta_12, "theta_13": self.theta_13,
                 "theta_23": self.theta_23, "delta_cp": self.delta_cp
            }

            try:
                # Temporarily update instance parameters
                self.A_up, self.k_up = current_A_up, current_k_up
                self.A_down, self.k_down = current_A_down, current_k_down
                self.L_u, self.L_d, self.L_s = current_L_u, current_L_d, current_L_s
                self.L_c, self.L_b, self.L_t = current_L_c, current_L_b, current_L_t
                self.gen_scale = current_gen_scale
                self.theta_12, self.theta_13 = current_theta_12, current_theta_13
                self.theta_23, self.delta_cp = current_theta_23, current_delta_cp

                # Calculate masses and errors
                masses = {}
                errors = {}
                total_error = 0
                for quark in self.quark_info:
                    ref_mass = self.quark_info[quark]["ref_mass"]
                    try:
                        masses[quark] = self.calculate_mass(quark)
                        if not np.isfinite(masses[quark]):
                            raise ValueError(f"Non-finite mass for {quark}")
                        if ref_mass > 1e-9:
                             error_term = ((masses[quark] - ref_mass) / ref_mass)**2
                        else:
                             error_term = (masses[quark] / 1e-3)**2

                        # Apply weights (Keep increased weight for "d")
                        weight = 1.0
                        if quark in ["c", "b", "d"]: weight = 5.0
                        elif quark == "t": weight = 20.0

                        errors[quark] = error_term
                        total_error += weight * error_term
                    except (ValueError, OverflowError) as e:
                        total_error += 1e10
                        errors[quark] = 1e10
                        masses[quark] = np.inf

                # Calculate CKM matrix and errors
                try:
                    ckm = self.calculate_ckm_matrix()
                    ckm_mag = np.abs(ckm)
                    if np.any(np.isnan(ckm_mag)) or np.any(np.isinf(ckm_mag)):
                        ckm_term_error = 1e10
                    else:
                        ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
                        ckm_errors = ((ckm_mag - self.ckm_exp) / ckm_exp_safe)**2
                        ckm_term_error = 10.0 * np.sum(ckm_errors)
                except (ValueError, OverflowError, np.linalg.LinAlgError) as e:
                     ckm_term_error = 1e10
                total_error += ckm_term_error

                # Regularization
                try:
                    # Regularize A and k towards initial guesses (or 0/1)
                    reg_exp = (current_A_up - 0.001)**2 + (current_k_up - 1.0)**2 + \
                              (current_A_down - 0.001)**2 + (current_k_down - 1.0)**2
                    current_Ls = np.array([current_L_u, current_L_d, current_L_s, current_L_c, current_L_b, current_L_t])
                    initial_Ls = np.array([0.1, 0.2, 0.5, 1.0, 2.0, 3.0])
                    reg_L = np.sum(np.square(current_Ls - initial_Ls))
                    reg_gen = np.sum(np.square(np.array(current_gen_scale) - 1.0))
                    reg_ckm = (current_theta_12 - 0.2)**2 + (current_theta_13 - 0.01)**2 + \
                              (current_theta_23 - 0.04)**2 + (current_delta_cp - 1.2)**2

                    regularization = 0.01 * (reg_exp + reg_L + reg_gen + reg_ckm)
                    if not np.isfinite(regularization):
                        regularization = 1e10
                except (ValueError, OverflowError) as e:
                     regularization = 1e10

                final_objective = total_error + regularization
                if np.isnan(final_objective) or np.isinf(final_objective):
                    return 1e20
                return final_objective

            finally:
                # Restore original instance parameters
                self.A_up, self.k_up = original_params["A_up"], original_params["k_up"]
                self.A_down, self.k_down = original_params["A_down"], original_params["k_down"]
                self.L_u, self.L_d, self.L_s = original_params["L_u"], original_params["L_d"], original_params["L_s"]
                self.L_c, self.L_b, self.L_t = original_params["L_c"], original_params["L_b"], original_params["L_t"]
                self.gen_scale = original_params["gen_scale"]
                self.theta_12, self.theta_13 = original_params["theta_12"], original_params["theta_13"]
                self.theta_23, self.delta_cp = original_params["theta_23"], original_params["delta_cp"]

        # Initial guess vector (17 parameters)
        initial_guess = np.array([
            self.A_up, self.k_up,         # 2 parameters
            self.A_down, self.k_down,     # 2 parameters
            self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t, # 6 parameters
            self.gen_scale[0], self.gen_scale[1], self.gen_scale[2], # 3 parameters
            self.theta_12, self.theta_13, self.theta_23, self.delta_cp # 4 parameters
        ])

        # Bounds list (should have 17 tuples)
        bounds = []
        # Bounds for A_up, k_up (Ensure A > 0, k can be pos/neg but start positive)
        bounds.append((1e-6, 1.0)) # A_up
        bounds.append((0.1, 10.0)) # k_up
        # Bounds for A_down, k_down
        bounds.append((1e-6, 1.0)) # A_down
        bounds.append((0.1, 10.0)) # k_down

        # Geodesic lengths (Ensure L > 0)
        bounds.append((1e-3, 0.5)) # L_u
        bounds.append((1e-3, 0.5)) # L_d
        bounds.append((0.1, 1.5)) # L_s
        bounds.append((0.5, 3.0)) # L_c
        bounds.append((1.0, 6.0)) # L_b
        bounds.append((2.0, 10.0)) # L_t

        # Generation scaling factors (Ensure scale > 0)
        bounds.append((1e-3, 20.0))  # gen_scale[0]
        bounds.append((1e-2, 100.0))  # gen_scale[1]
        bounds.append((0.1, 1000.0)) # gen_scale[2]

        # CKM parameters
        bounds.append((0.1, 0.3)) # theta_12
        bounds.append((0.001, 0.05)) # theta_13
        bounds.append((0.01, 0.1)) # theta_23
        bounds.append((0.0, 2*np.pi)) # delta_cp

        if len(bounds) != len(initial_guess):
             raise ValueError(f"Mismatch between parameters ({len(initial_guess)}) and bounds ({len(bounds)}).")

        # Perform optimization
        print(f"Starting optimization with L-BFGS-B (Exponential Model)...")
        result = minimize(objective, initial_guess, method="L-BFGS-B", bounds=bounds,
                          options={"maxiter": 50000, "maxfun": 50000, "ftol": 1e-9, "gtol": 1e-6, "disp": False})

        if result.success:
            print("Optimization successful.")
            optimized_params = result.x
            self.A_up = optimized_params[0]
            self.k_up = optimized_params[1]
            self.A_down = optimized_params[2]
            self.k_down = optimized_params[3]
            self.L_u = optimized_params[4]
            self.L_d = optimized_params[5]
            self.L_s = optimized_params[6]
            self.L_c = optimized_params[7]
            self.L_b = optimized_params[8]
            self.L_t = optimized_params[9]
            self.gen_scale = optimized_params[10:13]
            self.theta_12 = optimized_params[13]
            self.theta_13 = optimized_params[14]
            self.theta_23 = optimized_params[15]
            self.delta_cp = optimized_params[16]
            self.is_optimized = True
            final_objective_value = result.fun
            print(f"\n--- Final Optimized Parameter Vector (result.x) ---") # Add header
            print(result.x) # Print the vector
            print(f"----------------------------------------------------\n") # Add footer
        else:
            print(f"Optimization failed: {result.message}")
            self.is_optimized = False
            try:
                final_objective_value = objective(initial_guess)
            except Exception as e:
                print(f"Error calculating final objective with initial guess: {e}")
                final_objective_value = np.nan

        # Calculate final masses and errors
        final_masses = {}
        final_errors_percent = {}
        for quark in self.quark_info:
            ref_mass = self.quark_info[quark]["ref_mass"]
            pred_mass = self.calculate_mass(quark)
            final_masses[quark] = pred_mass
            if ref_mass > 1e-9:
                 if abs(ref_mass) > 1e-15:
                     final_errors_percent[quark] = abs((pred_mass - ref_mass) / ref_mass) * 100
                 else:
                     final_errors_percent[quark] = np.inf if abs(pred_mass) > 1e-9 else 0.0
            else:
                 final_errors_percent[quark] = abs(pred_mass) * 100

        # Calculate final CKM matrix and errors
        final_ckm = self.calculate_ckm_matrix()
        final_ckm_mag = np.abs(final_ckm)
        ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
        final_ckm_errors_percent = np.abs((final_ckm_mag - self.ckm_exp) / ckm_exp_safe) * 100

        # Store results
        self.results = {
            "A_up": self.A_up, "k_up": self.k_up, "A_down": self.A_down, "k_down": self.k_down,
            "L_u": self.L_u, "L_d": self.L_d, "L_s": self.L_s,
            "L_c": self.L_c, "L_b": self.L_b, "L_t": self.L_t,
            "gen_scale": self.gen_scale,
            "theta_12": self.theta_12, "theta_13": self.theta_13,
            "theta_23": self.theta_23, "delta_cp": self.delta_cp,
            "masses": final_masses,
            "errors_percent": final_errors_percent,
            "ckm": final_ckm_mag,
            "ckm_exp": self.ckm_exp,
            "ckm_errors_percent": final_ckm_errors_percent,
            "success": self.is_optimized,
            "message": result.message if hasattr(result, "message") else "Optimization not run or failed early",
            "final_objective_value": final_objective_value
        }

        return self.results

    def generate_report(self):
        """Generate a report of the model results (adapted for exponential model)."""
        if not self.results:
            print("Warning: No results found. Running optimization with initial parameters to generate report.")
            self.optimize_parameters()
            if not self.results:
                print("Error: Could not generate results.")
                return None

        report_path = "exponential_model_report.md"
        print(f"Generating report: {report_path}")
        try:
            with open(report_path, "w") as f:
                f.write("# Exponential Mass Model with CKM Matrix Report\n\n")
                f.write("## Optimization Status\n\n")

                # Fix for f-string dictionary access
                opt_success = self.results.get("success", False)
                opt_message = self.results.get("message", "N/A")
                final_obj = self.results.get("final_objective_value", np.nan)

                f.write(f"Success: {opt_success}\n")
                f.write(f"Message: {opt_message}\n")
                final_obj_str = f"{final_obj:.6e}" if not np.isnan(final_obj) else "N/A"
                f.write(f"Final Objective Function Value: {final_obj_str}\n\n")

                f.write("## Optimized Parameters (or Initial if Failed)\n\n")
                f.write("### Exponential Parameters (mass = A * exp(k*L))\n\n")
                A_up = self.results.get("A_up", np.nan)
                k_up = self.results.get("k_up", np.nan)
                A_down = self.results.get("A_down", np.nan)
                k_down = self.results.get("k_down", np.nan)

                A_up_str = f"{A_up:.6f}" if not np.isnan(A_up) else "NaN"
                k_up_str = f"{k_up:.6f}" if not np.isnan(k_up) else "NaN"
                A_down_str = f"{A_down:.6f}" if not np.isnan(A_down) else "NaN"
                k_down_str = f"{k_down:.6f}" if not np.isnan(k_down) else "NaN"

                f.write(f"Up-type A_u = {A_up_str}\n")
                f.write(f"Up-type k_u = {k_up_str}\n")
                f.write(f"Down-type A_d = {A_down_str}\n")
                f.write(f"Down-type k_d = {k_down_str}\n")

                f.write("\n### Generation Scaling Factors\n\n")
                gen_scale = self.results.get("gen_scale", [np.nan]*3)
                for i, s in enumerate(gen_scale):
                    s_str = f"{s:.6f}" if not np.isnan(s) else "NaN"
                    f.write(f"Generation {i+1}: {s_str}\n")

                f.write("\n### Geodesic Lengths\n\n")
                for q in ["u", "d", "s", "c", "b", "t"]:
                     l_val = self.results.get(f"L_{q}", np.nan)
                     l_str = f"{l_val:.6f}" if not np.isnan(l_val) else "NaN"
                     f.write(f"L_{q} = {l_str}\n")

                f.write("\n### CKM Matrix Parameters\n\n")
                theta_12 = self.results.get("theta_12", np.nan)
                theta_13 = self.results.get("theta_13", np.nan)
                theta_23 = self.results.get("theta_23", np.nan)
                delta_cp = self.results.get("delta_cp", np.nan)
                deg12 = np.degrees(theta_12) if not np.isnan(theta_12) else np.nan
                deg13 = np.degrees(theta_13) if not np.isnan(theta_13) else np.nan
                deg23 = np.degrees(theta_23) if not np.isnan(theta_23) else np.nan
                degCP = np.degrees(delta_cp) if not np.isnan(delta_cp) else np.nan

                theta_12_str = f"{theta_12:.6f}" if not np.isnan(theta_12) else "NaN"
                deg12_str = f"{deg12:.4f}" if not np.isnan(deg12) else "NaN"
                theta_13_str = f"{theta_13:.6f}" if not np.isnan(theta_13) else "NaN"
                deg13_str = f"{deg13:.4f}" if not np.isnan(deg13) else "NaN"
                theta_23_str = f"{theta_23:.6f}" if not np.isnan(theta_23) else "NaN"
                deg23_str = f"{deg23:.4f}" if not np.isnan(deg23) else "NaN"
                delta_cp_str = f"{delta_cp:.6f}" if not np.isnan(delta_cp) else "NaN"
                degCP_str = f"{degCP:.4f}" if not np.isnan(degCP) else "NaN"

                f.write(f"θ₁₂ = {theta_12_str} rad = {deg12_str}°\n")
                f.write(f"θ₁₃ = {theta_13_str} rad = {deg13_str}°\n")
                f.write(f"θ₂₃ = {theta_23_str} rad = {deg23_str}°\n")
                f.write(f"δ_CP = {delta_cp_str} rad = {degCP_str}°\n")

                f.write("\n## Mass Predictions at Reference Scales\n\n")
                f.write("| Quark | Reference Scale (GeV) | Predicted Mass (GeV) | Reference Value (GeV) | Error (%) |\n")
                f.write("|-------|----------------------|----------------------|----------------------|----------|\n")
                masses = self.results.get("masses", {}) or {}
                errors_percent = self.results.get("errors_percent", {}) or {}
                for quark in self.quark_info:
                    ref_scale = self.quark_info[quark]["ref_scale"]
                    ref_mass = self.quark_info[quark]["ref_mass"]
                    pred_mass = masses.get(quark, np.nan)
                    error = errors_percent.get(quark, np.nan)

                    pred_mass_str = f"{pred_mass:20.6f}" if not np.isnan(pred_mass) else "NaN"
                    error_str = f"{error:8.4f}" if not np.isnan(error) else "NaN"

                    f.write(f"| {quark:<5} | {ref_scale:20.4f} | {pred_mass_str:<20} | {ref_mass:20.6f} | {error_str:<8} |\n")

                f.write("\n## CKM Matrix\n\n")
                f.write("### Predicted CKM Matrix (Magnitudes)\n\n")
                f.write("```\n")
                ckm_pred = self.results.get("ckm", np.full((3,3), np.nan))
                for i in range(3):
                    line_parts = []
                    for j in range(3):
                        val = ckm_pred[i, j]
                        if not np.isnan(val):
                            line_parts.append(f"{val:.6f}")
                        else:
                            line_parts.append("NaN")
                    line = "[ " + " ".join(line_parts) + " ]\n"
                    f.write(line)
                f.write("```\n\n")

                f.write("### Experimental CKM Matrix (Magnitudes)\n\n")
                f.write("```\n")
                for i in range(3):
                    line = "[ " + " ".join([f"{self.ckm_exp[i, j]:.6f}" for j in range(3)]) + " ]\n"
                    f.write(line)
                f.write("```\n\n")

                f.write("### CKM Matrix Errors (%)\n\n")
                f.write("```\n")
                ckm_err_percent = self.results.get("ckm_errors_percent", np.full((3,3), np.nan))
                for i in range(3):
                    line_parts = []
                    for j in range(3):
                        err = ckm_err_percent[i, j]
                        if not np.isnan(err):
                            line_parts.append(f"{err:.4f}")
                        else:
                            line_parts.append("NaN")
                    line = "[ " + " ".join(line_parts) + " ]\n"
                    f.write(line)
                f.write("```\n\n")

                f.write("\n## Running Masses\n\n")
                # Fix for f-string dictionary access
                opt_success_rm = self.results.get("success", False)
                if opt_success_rm:
                     mu_values_report = [1.0, 2.0, 5.0, 10.0, self.mz, self.mt_mt]
                     try:
                         running_masses = self.calculate_running_masses(mu_values_report)
                         f.write("| μ (GeV) | m_u (GeV) | m_d (GeV) | m_s (GeV) | m_c (GeV) | m_b (GeV) | m_t (GeV) |\n")
                         f.write("|---------|-----------|-----------|-----------|-----------|-----------|----------|\n")
                         for i, mu in enumerate(running_masses["mu"]):
                              m_u = running_masses["u"][i]
                              m_d = running_masses["d"][i]
                              m_s = running_masses["s"][i]
                              m_c = running_masses["c"][i]
                              m_b = running_masses["b"][i]
                              m_t = running_masses["t"][i]

                              m_u_str = f"{m_u:9.6f}" if not np.isnan(m_u) else " NaN"
                              m_d_str = f"{m_d:9.6f}" if not np.isnan(m_d) else " NaN"
                              m_s_str = f"{m_s:9.6f}" if not np.isnan(m_s) else " NaN"
                              m_c_str = f"{m_c:9.6f}" if not np.isnan(m_c) else " NaN"
                              m_b_str = f"{m_b:9.6f}" if not np.isnan(m_b) else " NaN"
                              m_t_str = f"{m_t:9.6f}" if not np.isnan(m_t) else " NaN"

                              f.write(f"| {mu:7.4f} | {m_u_str:<9} | {m_d_str:<9} | {m_s_str:<9} | {m_c_str:<9} | {m_b_str:<9} | {m_t_str:<9} |\n")
                     except Exception as e:
                         f.write(f"Error calculating running masses for report: {e}\n")
                else:
                     f.write("Running masses not calculated (Optimization failed or not run).\n")

                f.write("\n## Strong Coupling Constant\n\n")
                mu_values_report = [1.0, 2.0, 5.0, 10.0, self.mz, self.mt_mt]
                try:
                    alpha_s_values = self.calculate_alpha_s_values(mu_values_report)
                    f.write("| μ (GeV) | α_s |\n")
                    f.write("|---------|------|\n")
                    for i, mu in enumerate(alpha_s_values["mu"]):
                         alpha = alpha_s_values["alpha_s"][i]
                         alpha_str = f"{alpha:.6f}" if not np.isnan(alpha) else " NaN"
                         f.write(f"| {mu:7.4f} | {alpha_str} |\n")
                except Exception as e:
                    f.write(f"Error calculating alpha_s values for report: {e}\n")

        except IOError as e:
            print(f"Error writing report file {report_path}: {e}")
            return None
        except Exception as e:
            print(f"An unexpected error occurred during report generation: {e}")
            return None

        return report_path
    # --- Plotting functions (adapted for exponential model) ---
    # Ensure they handle potential NaNs or missing data gracefully if optimization fails.

    def plot_running_masses(self):
        """Plot running masses (requires successful optimization)."""
        if not self.is_optimized:
            print("Warning: Cannot plot running masses. Optimization failed or not run.")
            return None

        fig, ax = plt.subplots(figsize=(10, 6))
        mu_values_plot = np.logspace(0, 3, 100) # 1 GeV to 1000 GeV
        try:
            running_masses = self.calculate_running_masses(mu_values_plot)
        except Exception as e:
            print(f"Error calculating running masses for plot: {e}")
            plt.close(fig)
            return None

        if running_masses is None:
            plt.close(fig)
            return None

        for quark in self.quark_info:
             if quark in running_masses and isinstance(running_masses[quark], list) and not np.all(np.isnan(running_masses[quark])):
                 valid_indices = ~np.isnan(running_masses[quark])
                 if np.any(valid_indices):
                     ax.loglog(running_masses["mu"][valid_indices], np.array(running_masses[quark])[valid_indices], label=f"{quark}")

        masses_pred = self.results.get("masses", {})
        for quark in self.quark_info:
             ref_scale = self.quark_info[quark]["ref_scale"]
             pred_mass = masses_pred.get(quark, np.nan)
             if not np.isnan(pred_mass) and not np.isnan(ref_scale):
                  ax.scatter([ref_scale], [pred_mass], marker="o", s=30, label=f"_{quark} pred")

        ax.set_title("Running Quark Masses (Predicted - Exponential Model)")
        ax.set_xlabel("Energy Scale μ (GeV)"); ax.set_ylabel("Running Mass (GeV)")
        ax.grid(True, which="both", linestyle="--", alpha=0.7)
        ax.legend(fontsize="small")
        ax.set_ylim(bottom=1e-3)

        plot_path = "exponential_running_masses_plot.png"
        try:
            plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error saving plot {plot_path}: {e}")
            plot_path = None
        finally:
            plt.close(fig)
        return plot_path

    def calculate_running_masses(self, mu_values):
        """Calculate running masses at different energy scales."""
        if not self.is_optimized:
            print("Warning: Parameters not optimized. Cannot calculate running masses accurately.")
            return {"mu": mu_values, **{q: [np.nan]*len(mu_values) for q in self.quark_info}}

        running_masses = {"mu": mu_values}
        for quark in self.quark_info:
            running_masses[quark] = []
            ref_mass_pred = self.results["masses"].get(quark, np.nan)
            if np.isnan(ref_mass_pred):
                print(f"Warning: Predicted reference mass for {quark} is NaN. Skipping running mass calculation.")
                running_masses[quark] = [np.nan] * len(mu_values)
                continue

            ref_scale = self.quark_info[quark]["ref_scale"]
            for mu in mu_values:
                if mu < 1.3: nf = 3
                elif mu < 4.2: nf = 4
                elif mu < 173.0: nf = 5
                else: nf = 6
                m_mu = self.running_mass(ref_mass_pred, ref_scale, mu, nf)
                running_masses[quark].append(m_mu)
        return running_masses

    def calculate_alpha_s_values(self, mu_values):
        """Calculate strong coupling constant at different energy scales."""
        alpha_s_values = {"mu": mu_values, "alpha_s": []}
        for mu in mu_values:
            try:
                alpha_val = self.alpha_s(mu)
                alpha_s_values["alpha_s"].append(alpha_val)
            except Exception as e:
                print(f"Error calculating alpha_s at mu={mu}: {e}")
                alpha_s_values["alpha_s"].append(np.nan)
        return alpha_s_values

    def plot_exponential_functions(self):
        """Plot the exponential mass generation functions (mass = A * exp(k*L))."""
        plot_path_base = "exponential_functions"
        if not self.results:
            print("Warning: No results found for plotting functions. Attempting optimization first.")
            self.optimize_parameters()
            if not self.results:
                print("Error: Cannot plot functions, results unavailable.")
                return None, None

        A_up = self.results.get("A_up", np.nan)
        k_up = self.results.get("k_up", np.nan)
        A_down = self.results.get("A_down", np.nan)
        k_down = self.results.get("k_down", np.nan)
        gen_scales = self.results.get("gen_scale", [1,1,1])
        Ls = {q: self.results.get(f"L_{q}", np.nan) for q in self.quark_info}
        pred_masses = self.results.get("masses", {})

        L_t_val = Ls.get("t", 5.0)
        L_max = L_t_val if not np.isnan(L_t_val) else 5.0
        L_values = np.linspace(0, L_max * 1.1, 200)

        # Calculate exponential values (handle potential NaNs in A/k)
        if np.isnan(A_up) or np.isnan(k_up):
            up_base_masses = np.full_like(L_values, np.nan)
        else:
            try:
                up_base_masses = A_up * np.exp(np.minimum(k_up * L_values, 700)) # Cap exponent
            except OverflowError:
                up_base_masses = np.full_like(L_values, np.inf)

        if np.isnan(A_down) or np.isnan(k_down):
            down_base_masses = np.full_like(L_values, np.nan)
        else:
            try:
                down_base_masses = A_down * np.exp(np.minimum(k_down * L_values, 700)) # Cap exponent
            except OverflowError:
                down_base_masses = np.full_like(L_values, np.inf)

        # --- Linear Scale Plot ---
        fig_lin, ax_lin = plt.subplots(figsize=(10, 6))
        plot_path_lin = None
        try:
            colors = ["r", "b"]; styles = ["--", "-.", "-"]
            labels_exp = [["Up Exp*g1", "Up Exp*g2", "Up Exp*g3"], ["Down Exp*g1", "Down Exp*g2", "Down Exp*g3"]]

            max_abs_mass_plot = 1.0
            if pred_masses:
                valid_masses_lin = [abs(m) for m in pred_masses.values() if m is not None and np.isfinite(m)]
                if valid_masses_lin: max_abs_mass_plot = max(valid_masses_lin) * 1.1

            for i in range(3):
                 mass_vals_up = up_base_masses * gen_scales[i]
                 ax_lin.plot(L_values, mass_vals_up, color=colors[0], linestyle=styles[i], alpha=0.6, label=labels_exp[0][i])
                 mass_vals_down = down_base_masses * gen_scales[i]
                 ax_lin.plot(L_values, mass_vals_down, color=colors[1], linestyle=styles[i], alpha=0.6, label=labels_exp[1][i])

            markers = ["o", "s", "^"]
            for quark, info in self.quark_info.items():
                q_L = Ls.get(quark, np.nan); q_mass = pred_masses.get(quark, np.nan)
                q_gen_idx = info["generation"] - 1; q_type_idx = 0 if info["type"] == "up" else 1
                if not np.isnan(q_L) and not np.isnan(q_mass):
                     ax_lin.scatter([q_L], [q_mass], color=colors[q_type_idx], marker=markers[q_gen_idx], s=60, label=f"{quark} (pred)", zorder=5)

            ax_lin.set_xlabel("Geodesic Length L")
            ax_lin.set_ylabel("Predicted Mass (GeV)")
            ax_lin.set_title("Exponential Mass Generation Functions (Scaled)")
            ax_lin.grid(True, linestyle="--", alpha=0.7)
            ax_lin.legend(fontsize="small", ncol=2)
            if max_abs_mass_plot > 0: ax_lin.set_ylim(bottom=-max_abs_mass_plot*0.1, top=max_abs_mass_plot)
            else: ax_lin.set_ylim(-1, 10)

            plot_path_lin = f"{plot_path_base}_plot.png"
            plt.savefig(plot_path_lin, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating linear exponential plot: {e}")
        finally:
            plt.close(fig_lin)

        # --- Log Scale Plot ---
        fig_log, ax_log = plt.subplots(figsize=(10, 6))
        plot_path_log = None
        try:
            colors = ["r", "b"]; styles = ["--", "-.", "-"]
            labels_exp = [["Up Exp*g1", "Up Exp*g2", "Up Exp*g3"], ["Down Exp*g1", "Down Exp*g2", "Down Exp*g3"]]

            for i in range(3):
                 mass_vals_up_log = np.maximum(up_base_masses * gen_scales[i], 1e-9)
                 ax_log.semilogy(L_values, mass_vals_up_log, color=colors[0], linestyle=styles[i], alpha=0.6, label=labels_exp[0][i])
                 mass_vals_down_log = np.maximum(down_base_masses * gen_scales[i], 1e-9)
                 ax_log.semilogy(L_values, mass_vals_down_log, color=colors[1], linestyle=styles[i], alpha=0.6, label=labels_exp[1][i])

            markers = ["o", "s", "^"]
            for quark, info in self.quark_info.items():
                q_L = Ls.get(quark, np.nan); q_mass = pred_masses.get(quark, np.nan)
                q_gen_idx = info["generation"] - 1; q_type_idx = 0 if info["type"] == "up" else 1
                if not np.isnan(q_L) and not np.isnan(q_mass) and q_mass > 1e-9:
                     ax_log.scatter([q_L], [q_mass], color=colors[q_type_idx], marker=markers[q_gen_idx], s=60, label=f"{quark} (pred)", zorder=5)

            ax_log.set_xlabel("Geodesic Length L")
            ax_log.set_ylabel("Predicted Mass (GeV) - Log Scale")
            ax_log.set_title("Exponential Mass Generation Functions (Scaled, Log Scale)")
            ax_log.grid(True, which="both", linestyle="--", alpha=0.7)
            ax_log.legend(fontsize="small", ncol=2)

            valid_masses_log = [m for m in pred_masses.values() if m is not None and np.isfinite(m) and m > 1e-9]
            if valid_masses_log:
                 min_mass_log = min(valid_masses_log); max_mass_log = max(valid_masses_log)
                 ax_log.set_ylim(bottom=min_mass_log * 0.1, top=max_mass_log * 10)
            else: ax_log.set_ylim(bottom=1e-3, top=1e3)

            plot_path_log = f"{plot_path_base}_log_plot.png"
            plt.savefig(plot_path_log, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating log exponential plot: {e}")
        finally:
            plt.close(fig_log)

        return plot_path_lin, plot_path_log

    def plot_geodesic_lengths(self):
        """Plot the geodesic lengths for all quarks."""
        if not self.results:
            print("Warning: No results found for plotting geodesic lengths. Attempting optimization first.")
            self.optimize_parameters()
            if not self.results: return None

        fig, ax = plt.subplots(figsize=(10, 6))
        plot_path = None
        try:
            quarks = ["u", "d", "s", "c", "b", "t"]
            lengths = [self.results.get(f"L_{q}", np.nan) for q in quarks]
            colors = ["red", "blue", "blue", "red", "blue", "red"]

            valid_indices = [i for i, l in enumerate(lengths) if not np.isnan(l)]
            if not valid_indices:
                ax.set_title("Geodesic Lengths - No valid data")
            else:
                quarks_valid = [quarks[i] for i in valid_indices]
                lengths_valid = [lengths[i] for i in valid_indices]
                colors_valid = [colors[i] for i in valid_indices]
                ax.bar(quarks_valid, lengths_valid, color=colors_valid)
                ax.set_title("Geodesic Lengths (Exponential Model)")
                ax.set_xlabel("Quark"); ax.set_ylabel("Optimized Geodesic Length (L)")
                ax.grid(True, axis="y", linestyle="--", alpha=0.7)

            plot_path = "exponential_geodesic_lengths_plot.png"
            plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating geodesic lengths plot: {e}")
        finally:
            plt.close(fig)
        return plot_path

    def plot_mass_hierarchy(self):
        """Plot the mass hierarchy for all quarks."""
        if not self.results:
            print("Warning: No results found for plotting mass hierarchy. Attempting optimization first.")
            self.optimize_parameters()
            if not self.results: return None

        fig, ax = plt.subplots(figsize=(10, 6))
        plot_path = None
        try:
            quarks = ["u", "d", "s", "c", "b", "t"]
            masses = [self.results.get("masses", {}).get(q, np.nan) for q in quarks]
            colors = ["red", "blue", "blue", "red", "blue", "red"]

            valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m) and m > 1e-9]
            if not valid_indices:
                ax.set_title("Quark Mass Hierarchy (Predicted - Exp Model) - No valid data")
            else:
                quarks_valid = [quarks[i] for i in valid_indices]
                masses_valid = [masses[i] for i in valid_indices]
                colors_valid = [colors[i] for i in valid_indices]
                ax.bar(quarks_valid, masses_valid, color=colors_valid)
                ax.set_yscale("log")
                ax.set_title("Quark Mass Hierarchy (Predicted - Exponential Model)")
                ax.set_xlabel("Quark"); ax.set_ylabel("Mass (GeV) - Log Scale")
                ax.grid(True, axis="y", which="both", linestyle="--", alpha=0.7)

            plot_path = "exponential_mass_hierarchy_plot.png"
            plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating mass hierarchy plot: {e}")
        finally:
            plt.close(fig)
        return plot_path

    def plot_ckm_matrix(self):
        """Plot the CKM matrix comparison."""
        if not self.results:
            print("Warning: No results found for plotting CKM matrix. Attempting optimization first.")
            self.optimize_parameters()
            if not self.results: return None

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        plot_path = None
        try:
            ckm_pred = self.results.get("ckm", np.full((3,3), np.nan))
            ckm_err_percent = self.results.get("ckm_errors_percent", np.full((3,3), np.nan))

            max_err = 20.0
            if not np.all(np.isnan(ckm_err_percent)): max_err = max(max_err, np.nanmax(ckm_err_percent))

            im1 = axes[0].imshow(ckm_pred, cmap="viridis", vmin=0, vmax=1)
            axes[0].set_title("Predicted CKM Matrix (Exp Model)")
            for i in range(3):
                for j in range(3):
                     val = ckm_pred[i, j]
                     text_color = "w" if (not np.isnan(val) and val < 0.5) else "k"
                     val_str = f"{val:.4f}" if not np.isnan(val) else "NaN"
                     axes[0].text(j, i, val_str, ha="center", va="center", color=text_color)

            im2 = axes[1].imshow(self.ckm_exp, cmap="viridis", vmin=0, vmax=1)
            axes[1].set_title("Experimental CKM Matrix")
            for i in range(3):
                for j in range(3):
                     val = self.ckm_exp[i, j]
                     text_color = "w" if val < 0.5 else "k"
                     axes[1].text(j, i, f"{val:.4f}", ha="center", va="center", color=text_color)

            im3 = axes[2].imshow(ckm_err_percent, cmap="hot", vmin=0, vmax=max_err)
            axes[2].set_title("Error Percentage (%) (Exp Model)")
            for i in range(3):
                for j in range(3):
                     err = ckm_err_percent[i, j]
                     text_color = "w" if (not np.isnan(err) and err > max_err*0.5) else "k"
                     err_str = f"{err:.2f}%" if not np.isnan(err) else "NaN"
                     axes[2].text(j, i, err_str, ha="center", va="center", color=text_color)

            for ax in axes:
                ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
                ax.set_xticklabels(["d", "s", "b"]); ax.set_yticklabels(["u", "c", "t"])
                ax.set_xlabel("Down-type"); ax.set_ylabel("Up-type")

            fig.colorbar(im1, ax=axes[0], label="Magnitude", shrink=0.7)
            fig.colorbar(im2, ax=axes[1], label="Magnitude", shrink=0.7)
            fig.colorbar(im3, ax=axes[2], label="Error (%)", shrink=0.7)
            plt.tight_layout()
            plot_path = "exponential_ckm_matrix_plot.png"
            plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating CKM matrix plot: {e}")
        finally:
            plt.close(fig)
        return plot_path

    def create_comprehensive_visualization(self):
        """Create a comprehensive visualization (adapted for exponential model)."""
        if not self.results:
            print("Warning: No results found for comprehensive visualization. Attempting optimization first.")
            self.optimize_parameters()
            if not self.results: return None

        fig = plt.figure(figsize=(15, 12))
        plot_path = None
        try:
            fig_title = "Comprehensive Model Results (Exponential Fit)"
            if not self.is_optimized: fig_title += " - OPTIMIZATION FAILED"
            fig.suptitle(fig_title, fontsize=16)

            # --- Plot 1: Running masses ---
            ax1 = fig.add_subplot(2, 2, 1)
            if self.is_optimized:
                 mu_values_plot = np.logspace(0, 3, 100)
                 try:
                     running_masses = self.calculate_running_masses(mu_values_plot)
                     if running_masses is not None:
                         for quark in self.quark_info:
                              if quark in running_masses and isinstance(running_masses[quark], list) and not np.all(np.isnan(running_masses[quark])):
                                  valid_indices = ~np.isnan(running_masses[quark])
                                  if np.any(valid_indices):
                                      ax1.loglog(running_masses["mu"][valid_indices], np.array(running_masses[quark])[valid_indices], label=f"{quark}")
                         masses_pred = self.results.get("masses", {})
                         for quark in self.quark_info:
                              ref_scale = self.quark_info[quark]["ref_scale"]
                              pred_mass = masses_pred.get(quark, np.nan)
                              if not np.isnan(pred_mass) and not np.isnan(ref_scale):
                                  ax1.scatter([ref_scale], [pred_mass], marker="o", s=30, label=f"_{quark} pred")
                         ax1.set_ylim(bottom=1e-3)
                         ax1.legend(fontsize="small")
                     else: ax1.text(0.5, 0.5, "Running masses calc failed", ha="center", va="center")
                 except Exception as e: ax1.text(0.5, 0.5, f"Error calculating running masses:\n{e}", ha="center", va="center")
                 ax1.set_title("Running Quark Masses (Predicted)")
                 ax1.set_xlabel("Energy Scale μ (GeV)"); ax1.set_ylabel("Running Mass (GeV)")
                 ax1.grid(True, which="both", linestyle="--", alpha=0.7)
            else:
                ax1.text(0.5, 0.5, "Running masses not plotted\n(Optimization failed)", ha="center", va="center")
                ax1.set_title("Running Quark Masses (Predicted)")

            # --- Plot 2: CKM matrix ---
            ax2 = fig.add_subplot(2, 2, 2)
            ckm_pred = self.results.get("ckm", np.full((3,3), np.nan))
            im = ax2.imshow(ckm_pred, cmap="viridis", vmin=0, vmax=1)
            ax2.set_title("Predicted CKM Matrix (Magnitudes)")
            for i in range(3):
                for j in range(3):
                     val = ckm_pred[i, j]
                     text_color = "w" if (not np.isnan(val) and val < 0.5) else "k"
                     val_str = f"{val:.4f}" if not np.isnan(val) else "NaN"
                     ax2.text(j, i, val_str, ha="center", va="center", color=text_color)
            ax2.set_xticks([0, 1, 2]); ax2.set_yticks([0, 1, 2])
            ax2.set_xticklabels(["d", "s", "b"]); ax2.set_yticklabels(["u", "c", "t"])
            ax2.set_xlabel("Down-type"); ax2.set_ylabel("Up-type")
            fig.colorbar(im, ax=ax2, label="Magnitude", shrink=0.8)

            # --- Plot 3: Exponential functions (Log Scale) ---
            ax3 = fig.add_subplot(2, 2, 3)
            A_up = self.results.get("A_up", np.nan); k_up = self.results.get("k_up", np.nan)
            A_down = self.results.get("A_down", np.nan); k_down = self.results.get("k_down", np.nan)
            gen_scales = self.results.get("gen_scale", [1,1,1])
            Ls = {q: self.results.get(f"L_{q}", np.nan) for q in self.quark_info}
            pred_masses = self.results.get("masses", {})
            L_t_val = Ls.get("t", 5.0); L_max = L_t_val if not np.isnan(L_t_val) else 5.0
            L_values = np.linspace(0, L_max * 1.1, 100)

            if np.isnan(A_up) or np.isnan(k_up): up_base = np.full_like(L_values, np.nan)
            else: up_base = A_up * np.exp(np.minimum(k_up * L_values, 700))
            if np.isnan(A_down) or np.isnan(k_down): down_base = np.full_like(L_values, np.nan)
            else: down_base = A_down * np.exp(np.minimum(k_down * L_values, 700))

            colors = ["r", "b"]; styles = ["--", "-.", "-"]
            for i in range(3):
                 ax3.semilogy(L_values, np.maximum(up_base * gen_scales[i], 1e-9), color=colors[0], linestyle=styles[i], alpha=0.6, label=f"Up Exp*g{i+1}")
                 ax3.semilogy(L_values, np.maximum(down_base * gen_scales[i], 1e-9), color=colors[1], linestyle=styles[i], alpha=0.6, label=f"Down Exp*g{i+1}")

            markers = ["o", "s", "^"]
            for quark, info in self.quark_info.items():
                q_L = Ls.get(quark, np.nan); q_mass = pred_masses.get(quark, np.nan)
                if not np.isnan(q_L) and not np.isnan(q_mass) and q_mass > 1e-9:
                     ax3.scatter([q_L], [q_mass], color=colors[0 if info["type"]=="up" else 1],
                                 marker=markers[info["generation"]-1], s=60, label=f"{quark} (pred)", zorder=5)

            ax3.set_xlabel("Geodesic Length L")
            ax3.set_ylabel("Predicted Mass (GeV) - Log Scale")
            ax3.set_title("Exponential Functions (Scaled)")
            ax3.grid(True, which="both", linestyle="--", alpha=0.7)
            ax3.legend(fontsize="small", ncol=2)
            valid_masses_log = [m for m in pred_masses.values() if m is not None and np.isfinite(m) and m > 1e-9]
            if valid_masses_log: ax3.set_ylim(min(valid_masses_log)*0.1, max(valid_masses_log)*10)
            else: ax3.set_ylim(1e-3, 1e3)

            # --- Plot 4: Mass hierarchy ---
            ax4 = fig.add_subplot(2, 2, 4)
            quarks = ["u", "d", "s", "c", "b", "t"]
            masses = [self.results.get("masses", {}).get(q, np.nan) for q in quarks]
            colors = ["red", "blue", "blue", "red", "blue", "red"]

            valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m) and m > 1e-9]
            if not valid_indices:
                ax4.text(0.5, 0.5, "Mass hierarchy not plotted\n(No valid positive mass data)", ha="center", va="center")
                ax4.set_title("Quark Mass Hierarchy (Predicted)")
            else:
                quarks_valid = [quarks[i] for i in valid_indices]
                masses_valid = [masses[i] for i in valid_indices]
                colors_valid = [colors[i] for i in valid_indices]
                ax4.bar(quarks_valid, masses_valid, color=colors_valid)
                ax4.set_yscale("log")
                ax4.set_title("Quark Mass Hierarchy (Predicted)")
                ax4.set_xlabel("Quark"); ax4.set_ylabel("Mass (GeV) - Log Scale")
                ax4.grid(True, axis="y", which="both", linestyle="--", alpha=0.7)

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plot_path = "exponential_comprehensive_visualization.png"
            plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        except Exception as e:
            print(f"Error generating comprehensive visualization: {e}")
        finally:
            plt.close(fig)
        return plot_path

# --- Main execution block ---
if __name__ == "__main__":
    # Create model instance
    model = ExponentialMassModel()

    # Optimize parameters
    results = model.optimize_parameters()

    # Generate report and plots (using optimized parameters if successful)
    print("\nGenerating report and plots (using optimized parameters if successful)...")
    report_path = model.generate_report()
    if report_path:
        print(f"Report generated: {report_path}")
    else:
        print("Report generation failed.")

    # List of plotting functions to call
    plot_functions = [
        model.plot_running_masses,
        model.plot_exponential_functions, # Updated function name
        model.plot_geodesic_lengths,
        model.plot_mass_hierarchy,
        model.plot_ckm_matrix,
        model.create_comprehensive_visualization
    ]

    for plot_func in plot_functions:
        try:
            plot_paths = plot_func()
            if plot_paths:
                 # Handle functions returning single or multiple paths
                 if isinstance(plot_paths, tuple):
                     for p in plot_paths:
                         if p: print(f"Plot generated: {p}")
                 elif isinstance(plot_paths, str):
                      print(f"Plot generated: {plot_paths}")
        except Exception as e:
            print(f"Error generating plot with {plot_func.__name__}: {e}")


    # --- Print summary (optional, report contains this info) ---
    print("\n--- Summary from Results ---")
    print(f"Optimization Success: {model.is_optimized}")

    # Fix for f-string dictionary access
    opt_message_summary = results.get("message", "N/A")
    print(f"Optimization Message: {opt_message_summary}")

    print("\nMass Predictions (% Error):")
    errors_percent = results.get("errors_percent", {})
    if errors_percent:
        for quark in model.quark_info:
            error = errors_percent.get(quark, np.nan)
            error_str = f"{error:.4f}" if not np.isnan(error) else "NaN"
            print(f"  {quark}: {error_str}%")
    else:
        print("  No mass error data available.")

    print("\nCKM Errors (%):")
    ckm_err_percent = results.get("ckm_errors_percent", np.full((3,3), np.nan))
    if not np.all(np.isnan(ckm_err_percent)):
        for i in range(3):
            line_parts = []
            for j in range(3):
                err = ckm_err_percent[i, j]
                if not np.isnan(err):
                    line_parts.append(f"{err:.4f}")
                else:
                    line_parts.append(" NaN ")
            joined_string = " ".join(line_parts)
            print(f"  [ {joined_string} ]")
    else:
        print("  No CKM error data available.")

    print("\nFinished.")



Starting optimization with L-BFGS-B (Exponential Model)...
Optimization successful.

--- Final Optimized Parameter Vector (result.x) ---
[1.37916635e-01 2.02149391e+00 1.50345964e-01 1.36355564e+00
 1.00000000e-03 5.00000000e-01 1.00000000e-01 1.40215343e+00
 2.25317237e+00 3.40258977e+00 1.56956571e-02 5.42289236e-01
 1.28990651e+00 2.26931783e-01 3.69000870e-03 4.18323078e-02
 1.14399557e+00]
----------------------------------------------------


Generating report and plots (using optimized parameters if successful)...
Generating report: exponential_model_report.md
Report generated: exponential_model_report.md
Plot generated: exponential_running_masses_plot.png
Plot generated: exponential_functions_plot.png
Plot generated: exponential_functions_log_plot.png
Plot generated: exponential_geodesic_lengths_plot.png
Plot generated: exponential_mass_hierarchy_plot.png
Plot generated: exponential_ckm_matrix_plot.png
Plot generated: exponential_comprehensive_visualization.png

--- Summary fro